# Notebook 5: Multi-Modal Fusion + Ablation Study

**Account:** A | **GPU:** T4 x2 | **Time:** ~10-12 hrs

**Datasets to attach:**
- `smartlms-openface-features` (OpenFace CSVs + Labels)
- `smartlms-scripts`
- Outputs from NB1 (as dataset, or re-upload trained_models/)
- Outputs from NB2 (videomae_features/ re-uploaded as dataset)
- Outputs from NB3 (vit_embeddings/ re-uploaded as dataset)
- Outputs from NB4 (audio_features/ re-uploaded as dataset)

**This is the FINAL notebook that produces your paper results!**

Runs multi-modal stacking ensemble + ablation study across all 6 streams.

In [ ]:
# ── Cell 1: Install dependencies ──
!pip install -q "numpy<2" xgboost optuna shap transformers

import torch, os, sys, shutil, glob, subprocess
print(f"PyTorch: {torch.__version__}, CUDA: {torch.cuda.is_available()}")
print(f"GPU: {torch.cuda.get_device_name(0)}")
print(f"Memory: {torch.cuda.get_device_properties(0).total_memory / 1e9:.1f} GB")
if torch.cuda.device_count() > 1:
    print(f"Multi-GPU: {torch.cuda.device_count()} GPUs — DataParallel enabled")

In [ ]:
# ── Cell 2: Copy scripts + setup ──
WORK = "/kaggle/working"
os.makedirs(f"{WORK}/app/ml", exist_ok=True)
open(f"{WORK}/app/__init__.py", "w").close()
open(f"{WORK}/app/ml/__init__.py", "w").close()

def find_script(name):
    hits = glob.glob(f"/kaggle/input/**/{name}", recursive=True)
    return hits[0] if hits else None

scripts = [
    "train_model_v2.py", "train_model_v3.py", "train_model_v4.py",
    "train_multimodal_v5.py", "extract_face_embeddings.py",
    "train_videomae.py", "extract_audio_features.py"
]
for s in scripts:
    found = find_script(s)
    if found:
        shutil.copy(found, f"{WORK}/app/ml/{s}")
        print(f"✓ {s} ← {found}")
    else:
        print(f"✗ NOT FOUND: {s}")

sys.path.insert(0, WORK)
os.chdir(WORK)

In [ ]:
# ── Cell 3: Auto-detect ALL data paths ──
import numpy as np

print("Attached datasets:")
for d in sorted(os.listdir("/kaggle/input")):
    print(f"  /kaggle/input/{d}/")

# ── OpenFace CSVs ──
csvs = glob.glob("/kaggle/input/**/openface_output/*.csv", recursive=True)
OPENFACE_DIR = os.path.dirname(csvs[0]) if csvs else None
print(f"\nOpenFace dir: {OPENFACE_DIR} ({len(csvs)} CSVs)")

# ── Labels ──
result = subprocess.run(["find", "/kaggle/input", "-name", "AllLabels.csv"], capture_output=True, text=True)
label_hits = [l for l in result.stdout.strip().split("\n") if l]
LABELS_DIR = os.path.dirname(label_hits[0]) if label_hits else None
DAISEE_DIR = os.path.dirname(LABELS_DIR) if LABELS_DIR else None
print(f"Labels dir: {LABELS_DIR}")

# ── VideoMAE features (from NB2 output) ──
vmae_hits = glob.glob("/kaggle/input/**/videomae_features/*.npy", recursive=True)
VIDEOMAE_DIR = os.path.dirname(vmae_hits[0]) if vmae_hits else None
print(f"VideoMAE features: {VIDEOMAE_DIR} ({len(vmae_hits)} files)")

# ── ViT embeddings (from NB3 output) ──
vit_hits = glob.glob("/kaggle/input/**/vit_embeddings/*.npy", recursive=True)
if not vit_hits:
    vit_hits = glob.glob("/kaggle/input/**/*.npy", recursive=True)
    vit_hits = [v for v in vit_hits if 'vit' in v.lower()]
VIT_DIR = os.path.dirname(vit_hits[0]) if vit_hits else None
print(f"ViT embeddings: {VIT_DIR} ({len(vit_hits)} files)")

# ── Audio features (from NB4 output) ──
audio_hits = glob.glob("/kaggle/input/**/audio_features/**/*.npy", recursive=True)
if not audio_hits:
    audio_hits = glob.glob("/kaggle/input/**/prosodic/*.npy", recursive=True)
AUDIO_DIR = os.path.dirname(os.path.dirname(audio_hits[0])) if audio_hits else None
print(f"Audio features: {AUDIO_DIR}")

# ── NB1 trained models (proba files) ──
proba_hits = glob.glob("/kaggle/input/**/proba_*.npy", recursive=True)
NB1_MODEL_DIR = os.path.dirname(proba_hits[0]) if proba_hits else None
print(f"NB1 models/probas: {NB1_MODEL_DIR} ({len(proba_hits)} proba files)")

# Copy NB1 trained_models to working dir so v5 can find them
os.makedirs("/kaggle/working/trained_models", exist_ok=True)
if NB1_MODEL_DIR:
    for f in glob.glob(os.path.join(NB1_MODEL_DIR, "*")):
        dst = f"/kaggle/working/trained_models/{os.path.basename(f)}"
        if not os.path.exists(dst):
            shutil.copy2(f, dst)
    print(f"✓ Copied NB1 model files to /kaggle/working/trained_models/")

In [ ]:
# ── Cell 4: Patch train_multimodal_v5.py with detected paths ──
v5_path = f"{WORK}/app/ml/train_multimodal_v5.py"
code = open(v5_path).read()

replacements = {
    'OPENFACE_DIR = "/kaggle/input/smartlms-openface/openface_output"': f'OPENFACE_DIR = "{OPENFACE_DIR}"' if OPENFACE_DIR else None,
    'DAISEE_DIR = "/kaggle/input/daisee-dataset"': f'DAISEE_DIR = "{DAISEE_DIR}"' if DAISEE_DIR else None,
    'VIT_DIR = "/kaggle/input/smartlms-vit-embeddings"': f'VIT_DIR = "{VIT_DIR}"' if VIT_DIR else None,
    'VIDEOMAE_DIR = "/kaggle/working/trained_models/videomae_features"': f'VIDEOMAE_DIR = "{VIDEOMAE_DIR}"' if VIDEOMAE_DIR else None,
    'AUDIO_DIR = "/kaggle/working/audio_features"': f'AUDIO_DIR = "{AUDIO_DIR}"' if AUDIO_DIR else None,
}

for old, new in replacements.items():
    if new:
        code = code.replace(old, new)
        print(f"✓ {old.split('=')[0].strip()} → {new.split('=')[1].strip()}")
    else:
        print(f"⚠ Skipped {old.split('=')[0].strip()} (not found in datasets)")

open(v5_path, 'w').write(code)

# Also patch v4 and v2
v4_path = f"{WORK}/app/ml/train_model_v4.py"
code = open(v4_path).read()
if OPENFACE_DIR: code = code.replace('OPENFACE_DIR = "/kaggle/input/smartlms-openface/openface_output"', f'OPENFACE_DIR = "{OPENFACE_DIR}"')
if DAISEE_DIR: code = code.replace('DAISEE_DIR = "/kaggle/input/daisee-dataset"', f'DAISEE_DIR = "{DAISEE_DIR}"')
open(v4_path, 'w').write(code)

v2_path = f"{WORK}/app/ml/train_model_v2.py"
if os.path.exists(v2_path) and DAISEE_DIR:
    code = open(v2_path).read()
    code = code.replace(r'C:\Users\revan\Downloads\DAiSEE', DAISEE_DIR)
    open(v2_path, 'w').write(code)

print("\n✅ All paths patched!")

In [ ]:
# ── Cell 5: Verify all modality data is accessible ──
from app.ml.train_model_v2 import load_labels

labels = load_labels(os.path.join(LABELS_DIR, "AllLabels.csv"))
print(f"Labels: {len(labels)} clips")

checks = [
    ("OpenFace CSVs", OPENFACE_DIR, "*.csv"),
    ("ViT embeddings", VIT_DIR, "*.npy"),
    ("Audio features", AUDIO_DIR, "**/*.npy"),
]
if VIDEOMAE_DIR:
    checks.append(("VideoMAE features", VIDEOMAE_DIR, "*.npy"))

for name, directory, pattern in checks:
    if directory and os.path.exists(directory):
        n = len(glob.glob(os.path.join(directory, pattern), recursive=True))
        status = "✓" if n > 0 else "✗"
        print(f"  {status} {name}: {n} files")
    else:
        print(f"  ✗ {name}: MISSING (dir={directory})")

proba_files = glob.glob("/kaggle/working/trained_models/proba_*.npy")
print(f"  {'✓' if proba_files else '✗'} NB1 probability outputs: {len(proba_files)} files")

print("\n✅ All available modalities will be used in fusion.")

In [ ]:
# ── Cell 6: Multi-Modal Stacking Fusion (~6 hrs) ──
# Combines all available modality streams into a single stacking ensemble
import sys
_orig_argv = sys.argv
sys.argv = ['train_multimodal_v5.py', '--mode', 'fusion_stack', '--epochs', '100']

from app.ml.train_multimodal_v5 import main as v5_main
v5_main()

sys.argv = _orig_argv

import torch, gc
gc.collect()
torch.cuda.empty_cache()
print("\n✅ Multi-modal fusion complete!")

In [ ]:
# ── Cell 7: Ablation Study (~4 hrs) ──
# Tests every combination of modalities — produces the paper ablation table
import sys, importlib
_orig_argv = sys.argv
sys.argv = ['train_multimodal_v5.py', '--mode', 'ablation']

# Reload to reset argparse state from previous cell
import app.ml.train_multimodal_v5 as v5_mod
importlib.reload(v5_mod)
v5_mod.main()

sys.argv = _orig_argv

import torch, gc
gc.collect()
torch.cuda.empty_cache()
print("\n✅ Ablation study complete!")

In [ ]:
# ── Cell 8: FINAL RESULTS ──
import json

print("=" * 80)
print("FINAL RESULTS — Multi-Modal Fusion v5")
print("=" * 80)

v3_baseline = {"XGBoost v3": 0.570, "BiLSTM v3": 0.537, "Ensemble v3": 0.563}

for rf in sorted(glob.glob("/kaggle/working/experiment_results/*.json")):
    print(f"\n📄 {os.path.basename(rf)}")
    with open(rf) as f:
        data = json.load(f)
    for key, val in data.items():
        if isinstance(val, dict):
            f1 = val.get('test_f1_macro') or val.get('cv_f1_macro_mean') or val.get('stacking_f1_macro') or val.get('f1_macro', 0)
            if f1:
                print(f"  {key}: F1m = {f1:.4f}")

print(f"\n{'='*80}")
print("v3 BASELINES for comparison:")
for k, v in v3_baseline.items():
    print(f"  {k}: F1m = {v:.3f}")
print(f"{'='*80}")
print("\n✅ DONE! Download and update your research paper with these results!")

In [ ]:
# ── Cell 9: Save all outputs ──
shutil.make_archive("/kaggle/working/final_models", 'zip', "/kaggle/working/trained_models")
shutil.make_archive("/kaggle/working/final_results", 'zip', "/kaggle/working/experiment_results")

for f in sorted(glob.glob("/kaggle/working/*.zip")):
    size_mb = os.path.getsize(f) / 1e6
    print(f"{os.path.basename(f)}: {size_mb:.1f} MB")

print("\n✅ Download from Output tab!")
print("These final models can be deployed to the SmartLMS backend.")